# 03_04 — Mapa SER proxy + EMT tiempo real

Ejemplo vivo de integración cartográfica:

- SER usa un intervalo SER de 30 minutos y `prob_aparcar_proxy` es una escala proxy relativa, no una probabilidad observada.
- EMT tiempo real usa la hora real de consulta a la API y no se redondea al intervalo SER.
- EMT no se predice y no tiene histórico en esta fase.
- La capa EMT es parcial: solo aparcamientos con dato vivo disponible y enlazado al inventario.
- `free_valid` mide plazas libres informadas por API, no ocupación SER ni probabilidad de aparcar.
- La integración es cartográfica, no un modelo conjunto SER+EMT.

In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError("No se encontró data_catalog.csv en los padres.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/hugo/TFM_parking_madrid')

In [2]:
from src.models.ser_parking_proxy import build_ser_parking_proxy_from_paths
from src.data.emt_realtime import build_emt_realtime_from_api
from src.visualization.parking_map import build_ser_prediction_map_with_emt_realtime

SER_PROXY_PATHS = {
    "m0_profiles_path": ROOT / "data/processed/core/ser/modeling/ser_m0_selected_profiles.parquet",
    "m0_metadata_path": ROOT / "data/processed/core/ser/modeling/ser_m0_selected_model_metadata.json",
    "capacidad_ser_path": ROOT / "data/processed/core/ser/ser_barrio_capacidad_anio.parquet",
    "autorizaciones_path": ROOT / "data/interim/ser/ser_autorizaciones/ser_autorizaciones_clean.parquet",
    "ivtm_cero_path": ROOT / "data/interim/ser/ser_padron_vehiculos_ivtm_barrio/ser_padron_vehiculos_ivtm_barrio_clean.parquet",
    "calendario_laboral_path": ROOT / "data/interim/contexto/contexto_calendario_laboral/contexto_calendario_laboral_clean.parquet",
}

INVENTORY_PATH = ROOT / "data/processed/core/emt/inventario_global_emt.parquet"
RAW_XML_OUTPUT_PATH = ROOT / "data/raw/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_aparcamientos_rotacionales_tiempo_real__latest.xml"
INTERIM_OUTPUT_PATH = ROOT / "data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_aparcamientos_rotacionales_tiempo_real_latest.parquet"
JOINED_OUTPUT_PATH = ROOT / "data/interim/emt/emt_aparcamientos_rotacionales_tiempo_real/emt_realtime_inventory_join_latest.parquet"
HTML_OUTPUT_PATH = ROOT / "reports/maps/mapa_integrado_ser_proxy_emt_tiempo_real.html"
PNG_EMT_REALTIME_OUTPUT_PATH = ROOT / "reports/figures/emt_tiempo_real/mapa_emt_tiempo_real_zoom.png"

## Escenario SER

La hora solicitada se toma en hora local de Madrid. El proxy SER se asigna al intervalo SER de 30 minutos correspondiente.

In [3]:
scenario_datetime = pd.Timestamp.now(tz="Europe/Madrid").tz_localize(None)
scenario_datetime

Timestamp('2026-07-08 17:02:56.871131')

In [4]:
ser_proxy = build_ser_parking_proxy_from_paths(
    **SER_PROXY_PATHS,
    scenario_datetime=scenario_datetime,
    ivtm_reference_year=2025,
    output_mode="operational",
    expected_n_barrios=65,
    strict=True,
)

ser_metadata_keys = [
    "scenario_datetime_requested",
    "scenario_datetime_used",
    "intervalo_inicio",
    "intervalo_fin",
    "intervalo_ajustado_30min",
]
pd.DataFrame(
    [{"campo": key, "valor": ser_proxy.metadata.get(key)} for key in ser_metadata_keys]
)

,campo,valor
0,scenario_datetime_requested,2026-07-08 17:02:56.871131
1,scenario_datetime_used,2026-07-08 17:00:00
2,intervalo_inicio,2026-07-08 17:00:00
3,intervalo_fin,2026-07-08 17:30:00
4,intervalo_ajustado_30min,True


## EMT tiempo real

La consulta EMT usa la hora real de llamada a la API. No se redondea al intervalo SER.

In [5]:
emt_realtime = build_emt_realtime_from_api(
    inventory_path=INVENTORY_PATH,
    raw_xml_output_path=RAW_XML_OUTPUT_PATH,
    interim_output_path=INTERIM_OUTPUT_PATH,
    joined_output_path=JOINED_OUTPUT_PATH,
)

emt_realtime.outputs

,output,path,exists,size_mb
0,raw_xml,/Users/hugo/TFM_parking_madrid/data/raw/emt/em...,True,0.049
1,interim_realtime,/Users/hugo/TFM_parking_madrid/data/interim/em...,True,0.016
2,joined_inventory,/Users/hugo/TFM_parking_madrid/data/interim/em...,True,0.041


In [6]:
emt_realtime.metadata

{'endpoint_url': 'https://servayto.madrid.es/MTPAR_WSINFO/InfoParking',
 'language': 'ES',
 'query_timestamp_utc': '2026-07-08T15:02:59.427736+00:00',
 'http_status': 200,
 'response_bytes': 51271,
 'soap_action': 'http://tempuri.org/iInfoParking/GetListParking',
 'n_realtime_rows': 75,
 'n_joined_rows': 85,
 'source': 'emt_realtime_api'}

In [7]:
emt_realtime.checks

,check_id,status,detail,critical
0,api_http_200,OK,HTTP 200,True
1,api_response_not_empty,OK,51271 bytes,True
2,api_xml_parseable,OK,XML parseable,True
3,realtime_not_empty,OK,75 filas,True
4,realtime_id_not_null,OK,nulos=0,True
5,realtime_id_unique,OK,ids_unicos=75,True
6,realtime_coordinates_parseable,OK,"lat_no_nulas=75, lon_no_nulas=75",False
7,realtime_free_raw_present_some,OK,free_raw_no_nulo=25,False
8,realtime_free_negative_count,OK,free_raw_negativo=0,False
9,inventory_file_exists,OK,/Users/hugo/TFM_parking_madrid/data/processed/...,True


In [8]:
emt_realtime.diagnostics["coverage_summary"]

,n_realtime_total,n_realtime_unique_ids,n_realtime_with_free_raw,n_realtime_with_live_free,n_realtime_free_negative,n_inventory_total,n_inventory_with_id_emt,n_inventory_matched_realtime,n_inventory_matched_live_free,n_realtime_only,n_inventory_only,pct_inventory_matched_realtime,pct_inventory_matched_live_free,pct_realtime_matched_inventory,pct_inventory_total_matched_realtime,pct_inventory_total_matched_live_free,pct_realtime_live_matched_inventory
0,75,75,25,25,0,85,83,66,21,9,17,0.795181,0.253012,0.88,0.776471,0.247059,0.84


## Mapa integrado

El HTML conserva la lectura SER proxy por barrio y añade una única capa EMT/off-street mixta. Los aparcamientos con `has_live_free == True` y dentro del área visual SER se simbolizan como disponibilidad viva; el resto queda como inventario estático de referencia.


In [9]:
map_result = build_ser_prediction_map_with_emt_realtime(
    root=ROOT,
    operational=ser_proxy.operational,
    scenario_metadata=ser_proxy.metadata,
    emt_realtime_joined=emt_realtime.joined,
    emt_realtime_metadata=emt_realtime.metadata,
    html_output_path=HTML_OUTPUT_PATH,
    png_emt_realtime_zoom_output_path=PNG_EMT_REALTIME_OUTPUT_PATH,
)

map_result.outputs

,output,path,exists,size_mb
0,html_ser_emt_tiempo_real_proxy,reports/maps/mapa_integrado_ser_proxy_emt_tiem...,True,19.584
1,png_emt_tiempo_real_zoom,reports/figures/emt_tiempo_real/mapa_emt_tiemp...,True,3.158


In [10]:
scenario = map_result.scenario
temporal_summary = pd.DataFrame(
    [
        {
            "componente": "SER proxy",
            "hora_solicitada_o_consulta": scenario.get("hora_solicitada"),
            "intervalo_o_dato_usado": scenario.get("intervalo_label"),
            "lectura": "Escala proxy relativa asignada al intervalo SER de 30 minutos.",
        },
        {
            "componente": "EMT tiempo real",
            "hora_solicitada_o_consulta": scenario.get("emt_query_timestamp_label"),
            "intervalo_o_dato_usado": scenario.get("emt_moment_label"),
            "lectura": "Disponibilidad viva parcial observada por API; no predicción.",
        },
    ]
)
temporal_summary

,componente,hora_solicitada_o_consulta,intervalo_o_dato_usado,lectura
0,SER proxy,17:02,17:00–17:30,Escala proxy relativa asignada al intervalo SE...
1,EMT tiempo real,08/07/2026 17:02:59,08/07/2026 17:00:40 – 08/07/2026 17:02:27,Disponibilidad viva parcial observada por API;...


In [11]:
emt_layer_diag = map_result.diagnostics["emt_realtime_layer"].iloc[0]
coverage_layers = pd.DataFrame(
    [
        {
            "capa": "barrios SER proxy",
            "n_total": len(map_result.layers["prediction_barrios"]),
            "n_visible_mapa": len(map_result.layers["prediction_barrios"]),
            "lectura": "Barrios del modelo SER coloreados por proxy.",
        },
        {
            "capa": "EMT inventario total",
            "n_total": len(map_result.layers["emt_inventory"]),
            "n_visible_mapa": len(map_result.layers["emt_map"]),
            "lectura": "Inventario EMT/off-street filtrado al área visual SER para el mapa.",
        },
        {
            "capa": "EMT tiempo real vivo enlazado",
            "n_total": len(map_result.layers["emt_realtime_live_all"]),
            "n_visible_mapa": len(map_result.layers["emt_realtime_live_all"]),
            "lectura": "Aparcamientos enlazados con free_valid antes del filtro espacial.",
        },
        {
            "capa": "EMT tiempo real vivo visible tras filtro espacial",
            "n_total": len(map_result.layers["emt_realtime_live_all"]),
            "n_visible_mapa": len(map_result.layers["emt_realtime_map"]),
            "lectura": "Subconjunto vivo dentro del límite SER + buffer visual.",
        },
        {
            "capa": "EMT inventario mostrado sin dato vivo",
            "n_total": len(map_result.layers["emt_map"]),
            "n_visible_mapa": int(emt_layer_diag["n_inventory_static_shown_without_live"]),
            "lectura": "Inventario estático no duplicado por la capa viva.",
        },
    ]
)
coverage_layers

,capa,n_total,n_visible_mapa,lectura
0,barrios SER proxy,65,65,Barrios del modelo SER coloreados por proxy.
1,EMT inventario total,85,74,Inventario EMT/off-street filtrado al área vis...
2,EMT tiempo real vivo enlazado,13,13,Aparcamientos enlazados con free_valid antes d...
3,EMT tiempo real vivo visible tras filtro espacial,13,13,Subconjunto vivo dentro del límite SER + buffe...
4,EMT inventario mostrado sin dato vivo,74,61,Inventario estático no duplicado por la capa v...


In [12]:
map_result.diagnostics["emt_realtime_spatial_filter"]

,n_emt_realtime_live_total,n_emt_realtime_live_in_visual_area,n_emt_realtime_live_outside_visual_area,visual_buffer_m
0,13,13,0,25


In [13]:
map_result.diagnostics["emt_realtime_outside_visual_area"]

,id_emt,nombre,free_valid,latitud,longitud


In [14]:
map_result.diagnostics["emt_realtime_layer"]

,n_joined_rows,n_live_joined_before_name_filter,n_live_excluded_name_mismatch,n_live_rows,n_live_rows_visible,n_inventory_static_shown_without_live,n_live_with_capacity_reference,n_live_with_pct_reference
0,85,21,8,13,13,61,13,13


In [15]:
map_result.diagnostics["emt_realtime_categories"]

,categoria_disponibilidad_emt,categoria_disponibilidad_emt_label,n_aparcamientos
0,baja,Baja: 0–29% libres sobre capacidad ref.,4
1,media,Media: 30–69% libres sobre capacidad ref.,9


In [16]:
map_result.checks.tail(12)

,check_id,status,detail,critical
20,prediction_barrio_key_not_null,OK,nulls=0,True
21,prediction_unique_barrio_key,OK,duplicates=0,True
22,prediction_expected_barrios,OK,barrios=65; expected=65,True
23,prediction_prob_not_null,OK,nulls=0,True
24,prediction_prob_range_0_1,OK,min=0.3753057960427284; max=0.8481493743406197,True
25,prediction_single_interval,OK,intervals=['2026-07-08 17:00:00'],True
26,prediction_join_all_barrios,OK,missing_after_join=0; map_not_prediction=[]; p...,True
27,prediction_no_missing_map_keys,OK,map_not_prediction=[],True
28,emt_realtime_live_layer_not_empty,OK,live_rows_visible=13; live_rows_total=13,False
29,emt_realtime_outside_visual_area,OK,outside=0; in_visual_area=13,False


## Lectura metodológica

- SER y EMT se muestran juntos, pero no forman un modelo conjunto.
- SER expresa facilidad proxy estimada para el intervalo SER usado.
- EMT expresa disponibilidad viva parcial observada por API en la hora real de consulta.
- La ausencia de dato EMT no significa aparcamiento lleno ni vacío.
- `free_valid` solo se interpreta cuando la API informa plazas libres no negativas.